In [0]:
from pyspark.sql import functions as F, Window

spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA pitchside")

stats = spark.table("cleaned_player_gw_stats")
fixtures = spark.table("cleaned_fixtures").select(
    F.col("id").alias("fx_id"), "home_team_id", "away_team_id"
)
teams = spark.table("cleaned_teams").select(
    F.col("id").alias("tm_id"),
    "strength_overall_home", "strength_overall_away",
    "strength_defence_home", "strength_defence_away",
)
players = spark.table("cleaned_players").select(
    F.col("id").alias("pl_id"),
    "player_position", "position_encoded", "team_id", "start_price_millions",
)

# 1) Opponent per stat row, then opponent strength
s = stats.join(fixtures, stats.fixture_id == fixtures.fx_id, "left")
s = s.withColumn(
    "opp_team_id",
    F.when(F.col("was_home"), F.col("away_team_id")).otherwise(F.col("home_team_id")),
)
s = s.join(teams, s.opp_team_id == teams.tm_id, "left")
s = s.withColumn(
    "opp_strength_overall",
    F.when(F.col("was_home"), F.col("strength_overall_away")).otherwise(F.col("strength_overall_home")),
).withColumn(
    "opp_strength_defence",
    F.when(F.col("was_home"), F.col("strength_defence_away")).otherwise(F.col("strength_defence_home")),
)

# 2) One row per player per gameweek
gw = s.groupBy("player_id", "season", "gameweek").agg(
    F.sum("total_points").alias("points"),
    F.sum("minutes").alias("minutes"),
    F.sum("expected_goals").alias("xg"),
    F.sum("expected_assists").alias("xa"),
    F.max("price_millions").alias("price_millions"),
    F.avg(F.col("was_home").cast("int")).alias("home_share"),
    F.avg("opp_strength_overall").alias("opp_strength_overall"),
    F.avg("opp_strength_defence").alias("opp_strength_defence"),
)

# 3) Rolling features use ONLY earlier gameweeks
w = Window.partitionBy("player_id", "season").orderBy("gameweek")
prev3 = w.rowsBetween(-3, -1)
prev5 = w.rowsBetween(-5, -1)
before = w.rowsBetween(Window.unboundedPreceding, -1)

feat = (
    gw.withColumn("form_3", F.avg("points").over(prev3))
      .withColumn("form_5", F.avg("points").over(prev5))
      .withColumn("minutes_3", F.avg("minutes").over(prev3))
      .withColumn("xg_3", F.avg("xg").over(prev3))
      .withColumn("xa_3", F.avg("xa").over(prev3))
      .withColumn("gws_before", F.count("points").over(before))
      .withColumn("has_history", F.col("gws_before") > 0)
)

# 4) Player attributes
feat = feat.join(players, feat.player_id == players.pl_id, "left").drop("pl_id")

# 5) Standardise opponent strength within each season (different scales per season)
ws = Window.partitionBy("season")
for c in ["opp_strength_overall", "opp_strength_defence"]:
    mean_c, std_c = F.avg(c).over(ws), F.stddev(c).over(ws)
    feat = feat.withColumn(c + "_z", F.when(std_c > 0, (F.col(c) - mean_c) / std_c))

# 6) A price of 0 is missing data, not a real price
feat = feat.withColumn("price_millions", F.when(F.col("price_millions") > 0, F.col("price_millions")))

# 7) Completeness flag
feat = feat.withColumn(
    "has_complete_inputs",
    F.col("price_millions").isNotNull()
    & F.col("xg_3").isNotNull()
    & F.col("opp_strength_defence_z").isNotNull()
    & F.col("has_history"),
)

feat = feat.select(
    "player_id", "season", "gameweek",
    "player_position", "position_encoded", "team_id",
    "start_price_millions", "price_millions", "home_share",
    "opp_strength_overall", "opp_strength_defence",
    "opp_strength_overall_z", "opp_strength_defence_z",
    "form_3", "form_5", "minutes_3", "xg_3", "xa_3",
    "gws_before", "has_history", "has_complete_inputs",
    F.col("points").alias("label_points"),
)

feat.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("features_player_gameweek")

out = spark.table("features_player_gameweek")
print("rows:", out.count())
out.groupBy("season").agg(
    F.sum(F.when(F.col("price_millions").isNull(), 1).otherwise(0)).alias("null_price_rows"),
    F.sum(F.col("has_complete_inputs").cast("int")).alias("complete_rows"),
    F.count("*").alias("rows"),
).show()
out.filter("gameweek = 1").select(F.count("form_3").alias("gw1_form_3_non_null")).show()

rows: 5809
+-------+---------------+-------------+----+
| season|null_price_rows|complete_rows|rows|
+-------+---------------+-------------+----+
|2025-26|              0|         2107|2847|
|2026-27|           1023|            0|2962|
+-------+---------------+-------------+----+

+-------------------+
|gw1_form_3_non_null|
+-------------------+
|                  0|
+-------------------+

